# Fine Tuned Model

Using twitter-roberta-base-sentiment as a base, fine tunes a model to predict whether a reply is in response to a liberal, conservative, or neutral account name

In [1]:
!pip install pyprojroot

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
from transformers import AutoTokenizer, AutoConfig
from transformers import AutoModelForSequenceClassification
from transformers import pipeline
from datasets import load_dataset

import polars as pl
import numpy as np
import time

from pyprojroot import here
from scipy.special import softmax
from ast import literal_eval

In [ ]:
# Setup model and tokenizer
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [10]:
# Load train and test data
train = load_dataset("drive/MyDrive/Data/train_reduced.parquet")
test = load_dataset("drive/MyDrive/Data/test_reduced.parquet")

FileNotFoundError: Couldn't find any data file at /Users/paulterrasi/Documents/Political-Signaling-Sentiment-Analysis/src/modeling/drive/MyDrive/Data/train_reduced.parquet.

In [5]:
# Preprocess text (username and link placeholders)
def preprocess_and_tokenize(text):
    new_text = []
    for t in text.split():
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    full_str = literal_eval(" ".join(new_text))
    return tokenizer(text, return_tensors='pt', truncation=True, max_length=512)

def inference_single(encoded_input) -> dict[str, np.float32]:
    output = model(**encoded_input)
    scores = softmax(output[0][0].detach().numpy())
    return {label: float(score) for label, score in zip(config.id2label.values(), scores)}

In [ ]:
train_encoded = preprocess_and_tokenize

In [4]:
for name, _ in model.named_parameters():
    print(name)

roberta.embeddings.word_embeddings.weight
roberta.embeddings.position_embeddings.weight
roberta.embeddings.token_type_embeddings.weight
roberta.embeddings.LayerNorm.weight
roberta.embeddings.LayerNorm.bias
roberta.encoder.layer.0.attention.self.query.weight
roberta.encoder.layer.0.attention.self.query.bias
roberta.encoder.layer.0.attention.self.key.weight
roberta.encoder.layer.0.attention.self.key.bias
roberta.encoder.layer.0.attention.self.value.weight
roberta.encoder.layer.0.attention.self.value.bias
roberta.encoder.layer.0.attention.output.dense.weight
roberta.encoder.layer.0.attention.output.dense.bias
roberta.encoder.layer.0.attention.output.LayerNorm.weight
roberta.encoder.layer.0.attention.output.LayerNorm.bias
roberta.encoder.layer.0.intermediate.dense.weight
roberta.encoder.layer.0.intermediate.dense.bias
roberta.encoder.layer.0.output.dense.weight
roberta.encoder.layer.0.output.dense.bias
roberta.encoder.layer.0.output.LayerNorm.weight
roberta.encoder.layer.0.output.LayerNorm

In [ ]:
train_preprocessed = train_data.map(preprocess_imdb, batched=True, fn_kwargs={'tokenizer': tokenizer})
preprocessed_dev_data = dev_data.map(preprocess_imdb, batched=True, fn_kwargs={'tokenizer': tokenizer})

In [11]:
test = test.with_columns(
    pl.col("content")
    .map_elements(inference_single, return_dtype=pl.Struct({'negative': pl.Float64, 'neutral': pl.Float64, 'positive': pl.Float64}))
    .alias("score")
).unnest("score")

train = train.with_columns(
    pl.col("content")
    .map_elements(inference_single, return_dtype=pl.Struct({'negative': pl.Float64, 'neutral': pl.Float64, 'positive': pl.Float64}))
    .alias("score")
).unnest("score")

In [15]:
calc_prediction = pl.when(pl.col("negative") > pl.col("neutral")).then(
        pl.when(pl.col("negative") > pl.col("positive"))
        .then(pl.lit("negative"))
        .otherwise(pl.lit("positive"))
    ).otherwise(pl.when(pl.col("neutral") > pl.col("positive"))
        .then(pl.lit("neutral"))
        .otherwise(pl.lit("positive"))
    ).alias("prediction")

train = train.with_columns(calc_prediction)
test = test.with_columns(calc_prediction)

In [28]:
train.group_by(["label", "prediction"]).agg(pl.len().alias("count")).with_columns((pl.col("count") / pl.col("count").sum().over("label")).alias("pct_of_label"))

label,prediction,count,pct_of_label
str,str,u32,f64
"""conversative""","""neutral""",7851,0.560505
"""neutral""","""positive""",5554,0.228193
"""liberal""","""neutral""",5689,0.550619
"""conversative""","""positive""",2466,0.176055
"""liberal""","""positive""",1817,0.175861
"""neutral""","""neutral""",13138,0.539792
"""neutral""","""negative""",5647,0.232014
"""conversative""","""negative""",3690,0.26344
"""liberal""","""negative""",2826,0.273519


In [29]:
test.group_by(["label", "prediction"]).agg(pl.len().alias("count")).with_columns((pl.col("count") / pl.col("count").sum().over("label")).alias("pct_of_label"))

label,prediction,count,pct_of_label
str,str,u32,f64
"""liberal""","""negative""",717,0.271694
"""neutral""","""negative""",1421,0.229083
"""conversative""","""negative""",946,0.265432
"""conversative""","""neutral""",1996,0.560045
"""liberal""","""neutral""",1450,0.549451
"""liberal""","""positive""",472,0.178856
"""conversative""","""positive""",622,0.174523
"""neutral""","""neutral""",3333,0.537321
"""neutral""","""positive""",1449,0.233597


In [31]:
train.group_by(["label"]).agg(pl.col("negative").mean(), pl.col("neutral").mean(), pl.col("positive").mean())

label,negative,neutral,positive
str,f64,f64,f64
"""neutral""",0.24788,0.488447,0.263673
"""liberal""",0.284649,0.502311,0.21304
"""conversative""",0.276757,0.506261,0.216982


In [32]:
test.group_by(["label"]).agg(pl.col("negative").mean(), pl.col("neutral").mean(), pl.col("positive").mean())

label,negative,neutral,positive
str,f64,f64,f64
"""liberal""",0.284045,0.505066,0.210889
"""conversative""",0.275335,0.507387,0.217278
"""neutral""",0.246498,0.486562,0.266941


In [16]:
train.write_parquet("drive/MyDrive/Data/train_reduced_twitter-roberta-sentiment.parquet")
test.write_parquet("drive/MyDrive/Data/test_reduced_twitter-roberta-sentiment.parquet")